In [19]:
import pandas as pd
import numpy as np
import re

In [20]:
pm_25 = pd.read_csv(r"C:\Users\Junayed\dhaka-aq\data\raw\pm25_raw.csv")

weather = pd.read_csv(r"C:\Users\Junayed\dhaka-aq\data\raw\weather_raw.csv")


###  Initial Dataset Inspection

Inspect the structure, column types, row counts, and null-value distributions of the raw air quality and meteorological datasets using `info()`.

In [21]:
print("PM2.5 raw:")
pm_25.info()
print()

print("Weather raw:")
weather.info()

PM2.5 raw:
<class 'pandas.DataFrame'>
RangeIndex: 26304 entries, 0 to 26303
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   time    26304 non-null  str    
 1   pm2_5   26304 non-null  float64
dtypes: float64(1), str(1)
memory usage: 411.1 KB

Weather raw:
<class 'pandas.DataFrame'>
RangeIndex: 1096 entries, 0 to 1095
Data columns (total 9 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   time                         1096 non-null   str    
 1   temperature_2m_max           1096 non-null   float64
 2   temperature_2m_min           1096 non-null   float64
 3   precipitation_sum            1096 non-null   float64
 4   wind_speed_10m_max           1096 non-null   float64
 5   wind_direction_10m_dominant  1096 non-null   int64  
 6   relative_humidity_2m_max     1096 non-null   int64  
 7   relative_humidity_2m_min     1096 non-null   int64  
 8   

### Datetime Parsing & Format Normalization

#### Transformations Applied

| Dataset | Original Column Type | Original Sample Format | Transformed Type | Resulting Timestamp Range |
| :--- | :--- | :--- | :--- | :--- |
| **`pm_25`** | `object` (`str`) | `'2023-10-01T00:00'` (ISO 8601 hourly) | `datetime64[ns]` | `2023-10-01 00:00:00` to `2026-09-30 23:00:00` |
| **`weather`** | `object` (`str`) | `'2023-10-01'` (Date string) | `datetime64[ns]` | `2023-10-01 00:00:00` to `2026-09-30 00:00:00` |

---

#### Key Observations & Structural Verification

- **Identical Temporal Coverage:** Both datasets cover the exact 3-year observation window from **October 1, 2023** through **September 30, 2026** (1,096 total calendar days).
- **Resolution Mismatch:**
  - `pm_25['time']` contains timestamps with hourly offsets (`00:00:00`, `01:00:00`, ..., `23:00:00`).
  - `weather['time']` contains normalized midnight timestamps (`00:00:00`).
- **Downstream Readiness:** The parsing standardizes temporal representations, enabling extraction of date-level keys (`.dt.normalize()` or `.dt.date`) to aggregate the 26,304 hourly PM2.5 readings into 1,096 daily summaries prior to merging.

In [22]:
pm_25["time"] = pd.to_datetime(pm_25["time"])
weather["time"] = pd.to_datetime(weather["time"])

In [23]:
print("PM2.5 time standard:")
pm_25["time"].head(10)


PM2.5 time standard:


0   2023-10-01 00:00:00
1   2023-10-01 01:00:00
2   2023-10-01 02:00:00
3   2023-10-01 03:00:00
4   2023-10-01 04:00:00
5   2023-10-01 05:00:00
6   2023-10-01 06:00:00
7   2023-10-01 07:00:00
8   2023-10-01 08:00:00
9   2023-10-01 09:00:00
Name: time, dtype: datetime64[us]

In [24]:
print("Weather time standard:")
weather["time"].head(10)

Weather time standard:


0   2023-10-01
1   2023-10-02
2   2023-10-03
3   2023-10-04
4   2023-10-05
5   2023-10-06
6   2023-10-07
7   2023-10-08
8   2023-10-09
9   2023-10-10
Name: time, dtype: datetime64[us]

### Hourly-to-Daily PM2.5 Aggregation & Feature Engineering

#### Processing Overview

- **Date Extraction:** Extracted the date component from `pm_25["time"]` and cast it back to `datetime64[ns]` to serve as a uniform daily grouping key.
- **Aggregation Logic:** Grouped the 26,304 hourly records by `date` to generate summary statistical features (`min`, `mean`, `max`, `median`) for each calendar day.
- **Precision Formatting:** Rounded all computed statistical metrics to 2 decimal places.

---

#### Aggregated Features Created

| Column Name | Metric | Source Column | Description |
| :--- | :--- | :--- | :--- |
| **`date`** | Temporal Key | `pm_25["time"]` | Grouping date (`datetime64[ns]`) |
| **`pm_25_min`** | Minimum | `pm_25["pm2_5"]` | Lowest recorded PM2.5 concentration during the 24-hour day |
| **`pm_25_mean`** | Arithmetic Mean | `pm_25["pm2_5"]` | Average daily PM2.5 exposure level (rounded to 2 decimal places) |
| **`pm_25_max`** | Maximum | `pm_25["pm2_5"]` | Peak PM2.5 concentration observed during the day |
| **`pm_25_median`** | Median | `pm_25["pm2_5"]` | Midpoint PM2.5 level (robust against hourly peak spikes) |

---

#### Output Verification

- **Row Count:** `1,096` records generated (`26,304` hourly rows $\div$ `24` hours/day).
- **Temporal Alignment:** The resulting row count matches the length of the `weather` dataset (`1,096` days), making both tables ready for a 1:1 daily merge on the date key.

In [25]:
pm_25["date"] = pd.to_datetime(pm_25["time"].dt.date)

pm25_daily = (
    pm_25.groupby('date')['pm2_5']
    .agg(pm_25_min = "min",
         pm_25_mean = "mean",
         pm_25_max = "max",
         pm_25_median = "median")
    .reset_index()
)

pm25_daily["pm_25_mean"] = pm25_daily["pm_25_mean"].round(2)
pm25_daily["pm_25_max"] = pm25_daily["pm_25_max"].round(2)
pm25_daily["pm_25_median"] = pm25_daily["pm_25_median"].round(2)
pm25_daily["pm_25_min"] = pm25_daily["pm_25_min"].round(2)

len(pm25_daily)

1096

In [26]:
pm25_daily.head(10)

,date,pm_25_min,pm_25_mean,pm_25_max,pm_25_median
0,2023-10-01,5.0,9.96,16.4,9.70
1,2023-10-02,7.8,11.37,18.3,10.95
2,2023-10-03,4.6,8.80,13.6,9.20
3,2023-10-04,3.9,8.72,13.1,8.95
4,2023-10-05,5.5,10.85,14.5,11.30
5,2023-10-06,10.0,19.42,33.6,15.15
6,2023-10-07,8.4,25.80,56.6,21.65
7,2023-10-08,11.6,31.01,53.7,28.45
8,2023-10-09,21.1,35.20,58.4,31.80
9,2023-10-10,33.5,66.62,131.8,59.05


### Sunshine Duration to Hours

#### Processing Steps

- **Unit Conversion:** Converted `sunshine_duration` from raw seconds to hours by dividing by 3,600 ($1 \text{ hr} = 3,600 \text{ s}$).
- **Precision Formatting:** Rounded the resulting `sunshine_hours` feature to 2 decimal places.
- **Redundant Feature Removal:** Dropped the original `sunshine_duration` column to avoid multicollinearity and reduce memory usage.

---

#### Feature Metadata & Sample Output

| Column Name | Metric Type | Units | Calculation | Value Range |
| :--- | :--- | :--- | :--- | :--- |
| **`sunshine_hours`** | Continuous float | Hours | $\text{sunshine\_duration} \div 3600$ | $0.00 \text{ to } 12.38 \text{ hrs}$ |

**First 10 Observations (`sunshine_hours`):**
`[10.38, 8.97, 8.70, 2.64, 0.86, 1.26, 9.70, 7.83, 10.03, 10.76]`

In [27]:
weather["sunshine_hours"] = (weather["sunshine_duration"] / 3600).round(2)
weather = weather.drop(columns=["sunshine_duration"])

weather["sunshine_hours"].head(10)

0    10.38
1     8.97
2     8.70
3     2.64
4     0.86
5     1.26
6     9.70
7     7.83
8    10.03
9    10.76
Name: sunshine_hours, dtype: float64

In [36]:
weather["sunshine_hours"].max()

np.float64(12.38)

In [37]:
weather["sunshine_hours"].min()

np.float64(0.0)

### Wind Direction Discretization

#### Transformation Overview

- **Objective:** Convert continuous azimuth angles (`wind_direction_10m_dominant`, $0^\circ \text{ to } 360^\circ$) into 8-point compass sector labels.
- **New Feature Added:** `wind_direction_lebel` (Object/String).

---

#### 8-Point Compass Mapping Standard

| Compass Sector | Azimuth Range |
| :--- | :--- |
| **N (North)** | $337.5^\circ \le \theta < 360^\circ$ or $0^\circ \le \theta < 22.5^\circ$ |
| **NE (Northeast)** | $22.5^\circ \le \theta < 67.5^\circ$ |
| **E (East)** | $67.5^\circ \le \theta < 112.5^\circ$ |
| **SE (Southeast)** | $112.5^\circ \le \theta < 157.5^\circ$ |
| **S (South)** | $157.5^\circ \le \theta < 202.5^\circ$ |
| **SW (Southwest)** | $202.5^\circ \le \theta < 247.5^\circ$ |
| **W (West)** | $247.5^\circ \le \theta < 292.5^\circ$ |
| **NW (Northwest)** | $292.5^\circ \le \theta < 337.5^\circ$ |

---

#### Execution Output

In [28]:
def degree_to_compass(deg):
    if pd.isna(deg):
        return np.nan
    deg = deg % 360

    if deg < 22.5:
        return "N"

    bins = [22.5, 67.5, 112.5, 157.5, 202.5, 247.5, 292.5, 337.5]
    labels = ["NE", "E", "SE", "S", "SW", "W", "NW"]

    for boundary, label in zip(bins[1:], labels):
        if deg < boundary:
            return label
        return "N"

weather["wind_direction_lebel"] = weather["wind_direction_10m_dominant"].apply(degree_to_compass)

weather["wind_direction_lebel"].head(10)


0    N
1    N
2    N
3    N
4    N
5    N
6    N
7    N
8    N
9    N
Name: wind_direction_lebel, dtype: str

### Rename the time column to date:

The column name of "date" named as "time" in weather dataset, renamed it to "Date" by .rename(columns = date)

In [29]:
rename_date = {
    "time": "date"
}

weather = weather.rename(columns= rename_date)
print(weather.columns)

Index(['date', 'temperature_2m_max', 'temperature_2m_min', 'precipitation_sum',
       'wind_speed_10m_max', 'wind_direction_10m_dominant',
       'relative_humidity_2m_max', 'relative_humidity_2m_min',
       'sunshine_hours', 'wind_direction_lebel'],
      dtype='str')


### Inner Merge on Date

#### Merge Operation Summary

- **Primary Join Key:** `date` (`datetime64[ns]`)
- **Join Strategy:** Inner join (`how="inner"`)
- **Left Table:** `pm25_daily` (1,096 daily aggregated PM2.5 records)
- **Right Table:** `weather` (1,096 daily meteorological records)

---

#### Output Verification & Alignment

| Metric | Left Table (`pm25_daily`) | Right Table (`weather`) | Merged Output (`df`) | Retention Rate |
| :--- | :--- | :--- | :--- | :--- |
| **Row Count** | 1,096 | 1,096 | **1,096** | 100% |
| **Column Count** | 5 | 9 (or 10 with wind label) | **13** (or 14) | Complete feature set |
| **Missing Values** | 0 | 0 | **0** | No dropped/unmatched rows |
| **Date Range** | 2023-10-01 to 2026-09-30 | 2023-10-01 to 2026-09-30 | **2023-10-01 to 2026-09-30** | Perfect 1:1 daily alignment |

---

#### Integrated Feature Schema

- **Temporal Index:** `date`
- **Air Quality Indicators:** `pm_25_min`, `pm_25_mean`, `pm_25_max`, `pm_25_median`
- **Meteorological Indicators:**
  - **Temperature:** `temperature_2m_max`, `temperature_2m_min`
  - **Precipitation:** `precipitation_sum`
  - **Wind Dynamics:** `wind_speed_10m_max`, `wind_direction_10m_dominant` (and `wind_direction_lebel` if retained)
  - **Moisture & Sunlight:** `relative_humidity_2m_max`, `relative_humidity_2m_min`, `sunshine_hours`

In [30]:
df = pd.merge(pm25_daily, weather, on="date", how="inner")
len(df)

1096

In [31]:
df.head(5)

,date,pm_25_min,pm_25_mean,pm_25_max,pm_25_median,temperature_2m_max,temperature_2m_min,precipitation_sum,wind_speed_10m_max,wind_direction_10m_dominant,relative_humidity_2m_max,relative_humidity_2m_min,sunshine_hours,wind_direction_lebel
0,2023-10-01,5.0,9.96,16.4,9.70,32.3,25.5,1.5,18.0,130,96,69,10.38,N
1,2023-10-02,7.8,11.37,18.3,10.95,31.6,25.7,2.7,18.7,134,95,70,8.97,N
2,2023-10-03,4.6,8.80,13.6,9.20,30.2,25.6,7.1,20.6,135,95,74,8.70,N
3,2023-10-04,3.9,8.72,13.1,8.95,29.0,25.1,30.0,22.9,139,95,80,2.64,N
4,2023-10-05,5.5,10.85,14.5,11.30,30.0,24.9,50.1,23.3,150,96,76,0.86,N


### Reordering & Schema Standardization

#### Column Arrangement Logic

The final DataFrame `df` is restructured to group related domain attributes sequentially:

1. **Temporal Identifier:** `date` (primary index key)
2. **Target / Air Quality Indicators:** `pm_25_mean`, `pm_25_max`, `pm_25_min`, `pm_25_median`
3. **Thermal Indicators:** `temperature_2m_max`, `temperature_2m_min`
4. **Moisture & Precipitation:** `relative_humidity_2m_max`, `relative_humidity_2m_min`, `precipitation_sum`
5. **Wind Dynamics:** `wind_speed_10m_max`, `wind_

In [34]:
cols_order = [
    "date", "pm_25_mean", "pm_25_max", "pm_25_min", "pm_25_median",
    "temperature_2m_max", "temperature_2m_min", 
    "relative_humidity_2m_max", "relative_humidity_2m_min",
    "precipitation_sum",
    "wind_speed_10m_max", "wind_direction_10m_dominant", "wind_direction_lebel",
    "sunshine_hours"
]

df = df[cols_order]

### Dataset Export & Pipeline Finalization

#### Export Specifications

- **Target File:** `dhaka_aq_clean.csv`
- **Serialization Format:** Comma-Separated Values (UTF-8)
- **Index Retention:** `index=False` (prevents auto-generated row numbers from polluting column schema)

---

#### Export Verification Summary

| Attribute | Specification | Status |
| :--- | :--- | :--- |
| **Row Count** | 1,096 records | Complete (3 full calendar years) |
| **Column Count** | 14 structured features | Verified |
| **Temporal Span** | 2023-10-01 to 2026-09-30 | Unbroken sequence |
| **Null / Missing Values** | 0 cells | Clean |
| **Approximate File Size** | $\approx 73.74\text{ KB}$ | Compact and optimized |


In [35]:
df.to_csv("dhaka_aq_clean.csv", index = False)

---

#### Final Pipeline Artifacts

The dataset is fully prepared for exploratory analysis, seasonal trend decomposition, or supervised modeling tasks (such as predicting `pm_25_mean` against meteorological drivers).